# External CNN vs DNN Validation

In [15]:
# =========================================================
# External Validation: CNN vs DNN (FIXED & ALIGNED VERSION)
# =========================================================
# This script:
# 1. Strictly follows the original External Validation logic
# 2. Applies identical preprocessing and semantic correction
# 3. Evaluates CNN and DNN fairly on the same CRLM cohort
# =========================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

In [16]:
# =========================================================
# Step 0: Global settings
# =========================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [17]:
# =========================================================
# Step 1: Path configuration (MATCH YOUR PROJECT)
# =========================================================
BASE_DIR = r"D:\结直肠癌肝转移Biomarker 诊断\投稿\Computational and Structural Biotechnology\Ready for Submit\For Submit\返修\返修数据与脚本"

CV_DIR = os.path.join(BASE_DIR, "cnn_vs_dnn_cv_results")
MODEL_DIR = os.path.join(CV_DIR, "ensemble_models")
GENE_FILE = os.path.join(BASE_DIR, "used_functional_genes_cv.txt")

CRLM_DATA_FILE = r"D:\结直肠癌肝转移Biomarker 诊断\新的策略\Autoencoder\validation_datasets\dat_crlm.csv"

OUTPUT_DIR = os.path.join(CV_DIR, "external_cnn_vs_dnn_fixed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Model directory:", MODEL_DIR)
print("External CRLM file:", CRLM_DATA_FILE)

Model directory: D:\结直肠癌肝转移Biomarker 诊断\投稿\Computational and Structural Biotechnology\Ready for Submit\For Submit\返修\返修数据与脚本\cnn_vs_dnn_cv_results\ensemble_models
External CRLM file: D:\结直肠癌肝转移Biomarker 诊断\新的策略\Autoencoder\validation_datasets\dat_crlm.csv


In [18]:
# =========================================================
# Step 2: Load functional genes
# =========================================================
with open(GENE_FILE) as f:
    genes = [g.strip() for g in f if g.strip()]

print(f"Loaded {len(genes)} functional genes")

Loaded 606 functional genes


In [ ]:
# =========================================================
# Step 3: Load external CRLM dataset
# =========================================================
crlm = pd.read_csv(CRLM_DATA_FILE, index_col=0)

# ✅ Gold-standard external label mapping (DO NOT use LabelEncoder)
y_true = crlm["status"].apply(
    lambda x: 1 if "metastasis" in str(x).lower() else 0
).values

print("External label distribution:")
print(pd.Series(y_true).value_counts())

# Build feature matrix (same order as training)
X_ext = pd.DataFrame(index=crlm.index, columns=genes)

available = [g for g in genes if g in crlm.columns]
missing = [g for g in genes if g not in crlm.columns]

X_ext[available] = crlm[available]
X_ext[missing] = 0.0

print(f"External samples: {X_ext.shape[0]}")
print(f"Missing genes filled with zero: {len(missing)}")

External label distribution:
0    18
1    18
Name: count, dtype: int64
External samples: 36
Missing genes filled with zero: 0


In [20]:
# =========================================================
# Step 4: Load ensemble models and scalers
# =========================================================
def load_ensemble(prefix):
    models, scalers = [], []
    for i in range(1, 6):
        model_path = os.path.join(MODEL_DIR, f"{prefix}_fold{i}.keras")
        scaler_path = os.path.join(MODEL_DIR, f"{prefix}_scaler{i}.pkl")
        models.append(tf.keras.models.load_model(model_path))
        scalers.append(joblib.load(scaler_path))
    return models, scalers

cnn_models, cnn_scalers = load_ensemble("cnn")
dnn_models, dnn_scalers = load_ensemble("dnn")

print("Loaded CNN and DNN ensembles")

Loaded CNN and DNN ensembles


In [21]:
# =========================================================
# Step 5: Ensemble prediction (shared function)
# =========================================================
def ensemble_predict(models, scalers, X, model_type):
    preds = []
    for model, scaler in zip(models, scalers):
        X_scaled = scaler.transform(X)
        if model_type == "cnn":
            X_scaled = np.expand_dims(X_scaled, axis=-1)
        preds.append(model.predict(X_scaled, verbose=0).flatten())
    return np.mean(preds, axis=0)

# Raw probabilities
cnn_raw = ensemble_predict(cnn_models, cnn_scalers, X_ext, "cnn")
dnn_raw = ensemble_predict(dnn_models, dnn_scalers, X_ext, "dnn")

# ✅ Semantic correction (CRITICAL – same as original script)
cnn_score = 1 - cnn_raw
dnn_score = 1 - dnn_raw

In [22]:
# =========================================================
# Step 6: Evaluation using Youden Index (robust version)
# =========================================================
def evaluate_external(y_true, y_score, name):
    auc = roc_auc_score(y_true, y_score)

    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    # ✅ Remove inf thresholds (CRITICAL FIX)
    valid = np.isfinite(thresholds)
    fpr, tpr, thresholds = fpr[valid], tpr[valid], thresholds[valid]

    youden = tpr - fpr
    idx = np.argmax(youden)
    thr = thresholds[idx]

    y_pred = (y_score >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "Model": name,
        "AUC": auc,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Best_Threshold": thr,
        "Confusion_Matrix": cm.tolist()
    }

cnn_res = evaluate_external(y_true, cnn_score, "CNN")
dnn_res = evaluate_external(y_true, dnn_score, "DNN")

In [23]:
# =========================================================
# Step 7: Output results
# =========================================================
results_df = pd.DataFrame([cnn_res, dnn_res])
print("\n=== External CNN vs DNN (FIXED) ===")
print(results_df)

results_df.to_csv(
    os.path.join(OUTPUT_DIR, "external_cnn_vs_dnn_summary.csv"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "cnn_report.txt"), "w") as f:
    f.write(classification_report(y_true, (cnn_score >= cnn_res["Best_Threshold"]).astype(int)))

with open(os.path.join(OUTPUT_DIR, "dnn_report.txt"), "w") as f:
    f.write(classification_report(y_true, (dnn_score >= dnn_res["Best_Threshold"]).astype(int)))

print(f"\n✅ External CNN vs DNN validation completed.")
print(f"Results saved to: {OUTPUT_DIR}")


=== External CNN vs DNN (FIXED) ===
  Model       AUC  Accuracy  Precision    Recall  Best_Threshold  \
0   CNN  0.950617  0.888889     0.9375  0.833333        0.042785   
1   DNN  0.938272  0.916667     1.0000  0.833333        0.147843   

     Confusion_Matrix  
0  [[17, 1], [3, 15]]  
1  [[18, 0], [3, 15]]  

✅ External CNN vs DNN validation completed.
Results saved to: D:\结直肠癌肝转移Biomarker 诊断\投稿\Computational and Structural Biotechnology\Ready for Submit\For Submit\返修\返修数据与脚本\cnn_vs_dnn_cv_results\external_cnn_vs_dnn_fixed
